# Experiment 05 — GRPO with verifiable arithmetic rewards

**Objective:** train Qwen3-1.7B using group-relative rewards for arithmetic answers. This is a small pipeline smoke test, **not** a claim of improved reasoning. Starts from the original base model (not DPO), uses a fresh LoRA adapter, and keeps Kaggle secrets out of the notebook. Kaggle T4 / Colab T4 supported.

Outputs: `grpo_report.json`, `adapter/` and a ZIP in the output directory. Reward: exact numeric answer inside `<answer>...</answer>` plus a small format reward. Baseline and post-training accuracy use a held-out deterministic arithmetic set.

In [ ]:
import os

# Must execute before importing torch/transformers or initializing CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%pip install -q -U \
    "transformers>=4.51,<5" \
    "trl>=0.19,<0.27" \
    "peft>=0.15,<0.19" \
    "accelerate>=1.4,<2" \
    "datasets>=3.5,<5" \
    "bitsandbytes>=0.45" \
    "huggingface-hub>=0.30,<1" \
    sentencepiece

In [ ]:
import json
import random
import re
import shutil
import time
from datetime import UTC, datetime
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

assert torch.cuda.is_available(), "Enable a GPU accelerator."
assert torch.cuda.device_count() == 1, "Restart runtime; exactly one GPU must be visible."
MODEL_ID = "Qwen/Qwen3-1.7B"
SEED = 42
TRAIN_EXAMPLES = 128
EVAL_EXAMPLES = 40
MAX_STEPS = 20
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 64
random.seed(SEED)
torch.manual_seed(SEED)
if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working/alignlab-grpo")
elif Path("/content").exists():
    OUTPUT_DIR = Path("/content/alignlab-grpo")
else:
    OUTPUT_DIR = Path("./alignlab-grpo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0), "Output:", OUTPUT_DIR)

## Deterministic arithmetic dataset

Train and evaluation examples are generated with independent seeds and checked for duplicate prompts. All tasks are integer addition/subtraction. The prompt requires a machine-checkable answer tag. This small synthetic benchmark is deliberately narrow.

In [ ]:
def make_examples(count, seed):
    rng = random.Random(seed)
    rows = []
    seen = set()
    while len(rows) < count:
        a, b = rng.randint(10, 999), rng.randint(10, 999)
        op = rng.choice(["+", "-"])
        question = f"Compute {a} {op} {b}."
        if question in seen:
            continue
        seen.add(question)
        answer = a + b if op == "+" else a - b
        prompt = f"{question} Return only the final integer in <answer>...</answer> tags."
        rows.append({"prompt": prompt, "answer": str(answer)})
    return rows


train_rows = make_examples(TRAIN_EXAMPLES, SEED)
eval_rows = make_examples(EVAL_EXAMPLES, SEED + 1000)
assert not ({r["prompt"] for r in train_rows} & {r["prompt"] for r in eval_rows})
train_dataset = Dataset.from_list(train_rows)
print("Train:", len(train_dataset), "Held-out:", len(eval_rows))
print(train_rows[0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant,
    device_map={"": 0},
    dtype=torch.float16,
)
model.config.use_cache = False
print("Loaded:", MODEL_ID, "Device:", model.device)

In [ ]:
ANSWER_RE = re.compile(r"<answer>\s*([+-]?\d+)\s*</answer>")


def completion_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        return "".join(str(item.get("content", "")) for item in completion)
    return str(completion)


def accuracy_reward(completions, answer, **kwargs):
    rewards = []
    for completion, expected in zip(completions, answer):
        match = ANSWER_RE.search(completion_text(completion))
        rewards.append(1.0 if match and match.group(1) == str(expected) else 0.0)
    return rewards


def format_reward(completions, **kwargs):
    return [0.1 if ANSWER_RE.fullmatch(completion_text(c).strip()) else 0.0 for c in completions]


assert accuracy_reward(["<answer>15</answer>"], ["15"]) == [1.0]
assert accuracy_reward(["<answer>14</answer>"], ["15"]) == [0.0]

## Held-out generation accuracy

Run with greedy decoding and identical prompts before and after training. This measures answer correctness and tag compliance, not general reasoning ability.

In [ ]:
def evaluate_accuracy(active_model, examples):
    active_model.eval()
    correct = 0
    formatted = 0
    outputs = []
    start = time.perf_counter()
    for row in examples:
        rendered = tokenizer.apply_chat_template(
            [{"role": "user", "content": row["prompt"]}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(rendered, return_tensors="pt")
        device = active_model.get_input_embeddings().weight.device
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            tokens = active_model.generate(
                **inputs,
                max_new_tokens=MAX_COMPLETION_LENGTH,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(
            tokens[0, inputs["input_ids"].shape[-1] :],
            skip_special_tokens=True,
        )
        match = ANSWER_RE.search(response)
        is_correct = bool(match and match.group(1) == row["answer"])
        is_formatted = bool(ANSWER_RE.fullmatch(response.strip()))
        correct += is_correct
        formatted += is_formatted
        outputs.append(
            {
                "prompt": row["prompt"],
                "expected": row["answer"],
                "response": response,
                "correct": is_correct,
            }
        )
    return {
        "accuracy": correct / len(examples),
        "format_rate": formatted / len(examples),
        "seconds": time.perf_counter() - start,
        "examples": outputs,
    }


baseline = evaluate_accuracy(model, eval_rows)
print("Baseline accuracy:", baseline["accuracy"])
print("Baseline format rate:", baseline["format_rate"])

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    per_device_train_batch_size=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=128,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=0.9,
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
assert training_args.n_gpu == 1
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[accuracy_reward, format_reward],
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)
print("Trainer initialized; visible GPUs:", torch.cuda.device_count())

In [ ]:
start = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - start
print("Training seconds:", round(train_seconds, 1))
print("Training metrics:", train_result.metrics)

In [ ]:
adapter_dir = OUTPUT_DIR / "adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
assert (adapter_dir / "adapter_config.json").is_file()
assert any((adapter_dir / f).is_file() for f in ("adapter_model.safetensors", "adapter_model.bin"))
trainer.model.config.use_cache = True
post = evaluate_accuracy(trainer.model, eval_rows)
print("Post-GRPO accuracy:", post["accuracy"])
print("Post-GRPO format rate:", post["format_rate"])

In [ ]:
report = {
    "experiment": "05_grpo_training",
    "timestamp_utc": datetime.now(UTC).isoformat(),
    "model": MODEL_ID,
    "initialization": "base model with fresh LoRA (not SFT or DPO)",
    "dataset": "deterministic synthetic integer addition/subtraction",
    "train_examples": TRAIN_EXAMPLES,
    "eval_examples": EVAL_EXAMPLES,
    "seed": SEED,
    "max_steps": MAX_STEPS,
    "num_generations": NUM_GENERATIONS,
    "train_seconds": train_seconds,
    "train_metrics": train_result.metrics,
    "baseline": baseline,
    "grpo": post,
    "accuracy_delta": post["accuracy"] - baseline["accuracy"],
    "torch_version": torch.__version__,
}
report_path = OUTPUT_DIR / "grpo_report.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Report:", report_path)
print("Adapter:", adapter_dir)
print("Archive:", archive)

## Interpretation and next steps

This smoke test checks whether the GRPO pipeline works and whether arithmetic accuracy changes. Twenty updates and 128 examples are not enough to establish general reasoning gains. If rewards are almost always identical within groups, GRPO may have little learning signal. Before making claims, expand held-out evaluation, compare multiple seeds, inspect reward variance, and run a stronger math benchmark. Publish the adapter through GitHub Actions after successful Kaggle execution; **do not place HF_TOKEN inside Kaggle**.